In [7]:
import pandas as pd
import numpy as np
import  requests
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
def data_load():
    url = 'https://www.remoteok.com/remote-dev-jobs.json'
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept' : 'application/json'}

    response = requests.get(url, headers = headers)
    jobs = response.json()


    jobber = []
    for job in jobs[1:]:
        jobber.append({
            'title' : job.get('position'),
            'company' : job.get('company'),
            'skills' : job.get('tags'),
            'link' : job.get('url')
        })

    df = pd.DataFrame(jobber)
    df['text'] = df['title'] + '' + df['skills']

    return df

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')


def modelling(df):
    embeddings = model.encode(df['text'].tolist(), show_progress_bar=False)
    return embeddings



def job_recommend(user_input, df, embeddings):
    user_embedding = model.encode(user_input)
    similarity = cosine_similarity(user_embedding, embeddings)[0]

    top_matches = np.argsort(similarity)[::-1][:5]

    results = []
    for idx in top_matches:
        job = df.iloc[idx]
        score = similarity[idx]
        results.append((job, score))

    return results

c:\Users\USER\Desktop\Job_Recommender\jobber\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4784.25it/s]
BertModel LOAD RE